In [8]:
"""
==============================================================
Bluestock Mutual Fund ETL Pipeline
==============================================================

Author : Arunima Jain

Description:
------------
This ETL Pipeline performs:

1. Read raw datasets
2. Clean datasets
3. Handle missing values
4. Remove duplicates
5. Convert data types
6. Save processed datasets

==============================================================
"""

# ============================================================
# Import Libraries
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# Project Directories
# ============================================================

PROJECT_DIR = Path(
    r"C:\Users\aruni\OneDrive\Documents\Project Mutual Fund Analysis"
)

RAW_DIR = PROJECT_DIR / "data" / "raw"

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

# Create processed folder if it doesn't exist
PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# Start Message
# ============================================================

print("=" * 70)
print(" Bluestock Mutual Fund ETL Pipeline Started ")
print("=" * 70)

# ============================================================
# Helper Functions
# ============================================================

def save_dataset(df, filename):
    """
    Save cleaned dataframe to processed folder.
    """

    output_path = PROCESSED_DIR / filename

    df.to_csv(
        output_path,
        index=False
    )

    print(f"✓ Saved : {filename}")


def remove_duplicates(df):
    """
    Remove duplicate rows.
    """

    before = len(df)

    df = df.drop_duplicates()

    after = len(df)

    print(f"Removed {before-after} duplicate rows")

    return df


def convert_date(df, column_name):
    """
    Convert date column into datetime.
    """

    df[column_name] = pd.to_datetime(
        df[column_name],
        errors="coerce"
    )

    return df


def convert_numeric(df, column_name):
    """
    Convert column into numeric.
    """

    df[column_name] = pd.to_numeric(
        df[column_name],
        errors="coerce"
    )

    return df


def print_shape(df, name):
    """
    Display dataframe size.
    """

    print(f"{name} Shape : {df.shape}")


# ============================================================
# ETL Begins
# ============================================================

try:

    print("\n")

    print("=" * 70)
    print("Reading Raw CSV Files")
    print("=" * 70)

    # --------------------------------------------------------
    # Load CSV Files
    # --------------------------------------------------------

    fund_master = pd.read_csv(
        RAW_DIR / "fund_master.csv"
    )

    nav_history = pd.read_csv(
        RAW_DIR / "nav_history.csv"
    )

    investor_transactions = pd.read_csv(
        RAW_DIR / "investor_transactions.csv"
    )

    scheme_performance = pd.read_csv(
        RAW_DIR / "scheme_performance.csv"
    )

    aum_history = pd.read_csv(
        RAW_DIR / "aum_history.csv"
    )

    sector_allocation = pd.read_csv(
        RAW_DIR / "sector_allocation.csv"
    )

    print("✓ All CSV files loaded successfully.\n")

    # --------------------------------------------------------
    # Display Dataset Shapes
    # --------------------------------------------------------

    print_shape(fund_master, "Fund Master")

    print_shape(nav_history, "NAV History")

    print_shape(
        investor_transactions,
        "Investor Transactions"
    )

    print_shape(
        scheme_performance,
        "Scheme Performance"
    )

    print_shape(
        aum_history,
        "AUM History"
    )

    print_shape(
        sector_allocation,
        "Sector Allocation"
    )

    print("\n")



    # ============================================================
    # CLEAN FUND MASTER
    # ============================================================

    print("=" * 70)
    print("Cleaning Fund Master")
    print("=" * 70)

    # Remove duplicates
    fund_master = remove_duplicates(fund_master)

    # Convert Scheme Code to integer
    fund_master["Scheme Code"] = (
        pd.to_numeric(
            fund_master["Scheme Code"],
            errors="coerce"
        )
    )

    fund_master.dropna(
        subset=["Scheme Code"],
        inplace=True
    )

    fund_master["Scheme Code"] = (
        fund_master["Scheme Code"]
        .astype(int)
    )

    # Convert NAV
    fund_master["Net Asset Value"] = (
        pd.to_numeric(
            fund_master["Net Asset Value"],
            errors="coerce"
        )
    )

    # Convert Date
    fund_master["Date"] = pd.to_datetime(
        fund_master["Date"],
        errors="coerce"
    )

    # Remove rows with missing Scheme Name
    fund_master.dropna(
        subset=["Scheme Name"],
        inplace=True
    )

    # Fill missing text values
    text_columns = [
        "Fund House",
        "Category",
        "Sub Category"
    ]

    for col in text_columns:

        fund_master[col] = (
            fund_master[col]
            .fillna("Unknown")
        )

    print(fund_master.info())

    print("\nFund Master cleaned successfully.\n")

    # Save
    save_dataset(
        fund_master,
        "fund_master_clean.csv"
    )

    # ============================================================
    # CLEAN NAV HISTORY
    # ============================================================

    print("=" * 70)
    print("Cleaning NAV History")
    print("=" * 70)

    nav_history = remove_duplicates(nav_history)

    # Date
    nav_history["Date"] = pd.to_datetime(
        nav_history["Date"],
        errors="coerce"
    )

    # NAV
    nav_history["NAV"] = pd.to_numeric(
        nav_history["NAV"],
        errors="coerce"
    )

    # Scheme Code
    nav_history["Scheme Code"] = (
        pd.to_numeric(
            nav_history["Scheme Code"],
            errors="coerce"
        )
    )

    nav_history.dropna(
        subset=[
            "Date",
            "NAV",
            "Scheme Code"
        ],
        inplace=True
    )

    nav_history["Scheme Code"] = (
        nav_history["Scheme Code"]
        .astype(int)
    )

    # Remove invalid NAV
    nav_history = nav_history[
        nav_history["NAV"] > 0
    ]

    # Sort values
    nav_history = (
        nav_history
        .sort_values(
            [
                "Scheme Code",
                "Date"
            ]
        )
        .reset_index(drop=True)
    )

    # Forward fill NAV (holiday/weekend handling)
    nav_history["NAV"] = (
        nav_history
        .groupby("Scheme Code")["NAV"]
        .ffill()
    )

    # Fill missing text values
    nav_history["Scheme Name"] = (
        nav_history["Scheme Name"]
        .fillna("Unknown")
    )

    nav_history["Fund House"] = (
        nav_history["Fund House"]
        .fillna("Unknown")
    )

    nav_history["Sub Category"] = (
        nav_history["Sub Category"]
        .fillna("Unknown")
    )

    print(nav_history.info())

    print("\nNAV History cleaned successfully.\n")

    save_dataset(
        nav_history,
        "nav_history_clean.csv"
    )

    print("\nFirst two datasets completed successfully.\n")


    # ============================================================
    # CLEAN INVESTOR TRANSACTIONS
    # ============================================================

    print("=" * 70)
    print("Cleaning Investor Transactions")
    print("=" * 70)

    investor_transactions = remove_duplicates(
        investor_transactions
    )

    # Convert Transaction Date
    investor_transactions["Transaction_Date"] = pd.to_datetime(
        investor_transactions["Transaction_Date"],
        errors="coerce"
    )
    
    # Convert numeric columns
    numeric_cols = [
        "Scheme_Code",
        "Units",
        "NAV",
        "Amount"
    ]
    
    for col in numeric_cols:
        investor_transactions[col] = pd.to_numeric(
            investor_transactions[col],
            errors="coerce"
        )
    
    # Generate Investor IDs
    investor_transactions["Investor_ID"] = [
        f"INV{100001+i}"
        for i in range(len(investor_transactions))
    ]
    
    # Remove rows only if essential columns are missing
    investor_transactions.dropna(
        subset=[
            "Scheme_Code",
            "Transaction_Date",
            "Amount"
        ],
        inplace=True
    )
    
    # Convert Scheme Code to integer
    investor_transactions["Scheme_Code"] = (
        investor_transactions["Scheme_Code"]
        .astype(int)
    )
    
    # Remove invalid transaction amounts
    investor_transactions = investor_transactions[
        investor_transactions["Amount"] > 0
    ]
    
    # Fill missing categorical values
    investor_transactions["Transaction_Type"] = (
        investor_transactions["Transaction_Type"]
        .fillna("Unknown")
    )
    
    investor_transactions["State"] = (
        investor_transactions["State"]
        .fillna("Unknown")
    )
    
    investor_transactions["KYC_Status"] = (
        investor_transactions["KYC_Status"]
        .fillna("Pending")
    )
    
    # Sort transactions
    investor_transactions = investor_transactions.sort_values(
        "Transaction_Date"
    )

    print(investor_transactions.info())

    print("\nInvestor Transactions cleaned successfully.\n")

    save_dataset(
        investor_transactions,
        "investor_transactions_clean.csv"
    )

    # ============================================================
    # CLEAN SCHEME PERFORMANCE
    # ============================================================

    print("=" * 70)
    print("Cleaning Scheme Performance")
    print("=" * 70)

    scheme_performance = remove_duplicates(
        scheme_performance
    )

    # Scheme Code
    scheme_performance["Scheme Code"] = pd.to_numeric(
        scheme_performance["Scheme Code"],
        errors="coerce"
    )

    scheme_performance.dropna(
        subset=["Scheme Code"],
        inplace=True
    )

    scheme_performance["Scheme Code"] = (
        scheme_performance["Scheme Code"]
        .astype(int)
    )

    # Numeric Columns

    numeric_columns = [

        "Latest_NAV",

        "Return_1Y",

        "Return_3Y",

        "Return_5Y",

        "Expense_Ratio",

        "AUM_Cr"

    ]

    for col in numeric_columns:

        scheme_performance[col] = pd.to_numeric(

            scheme_performance[col],

            errors="coerce"

        )

    # Fill missing values

    scheme_performance["Risk_Level"] = (

        scheme_performance["Risk_Level"]

        .fillna("Moderate")

    )

    scheme_performance["Fund House"] = (

        scheme_performance["Fund House"]

        .fillna("Unknown")

    )

    scheme_performance["Category"] = (

        scheme_performance["Category"]

        .fillna("Unknown")

    )

    scheme_performance["Scheme Name"] = (

        scheme_performance["Scheme Name"]

        .fillna("Unknown")

    )

    # Remove rows having missing NAV

    scheme_performance.dropna(

        subset=["Latest_NAV"],

        inplace=True

    )

    # Remove duplicate schemes

    scheme_performance = (

        scheme_performance

        .drop_duplicates(

            subset="Scheme Code"

        )

    )

    print(scheme_performance.info())

    print("\nScheme Performance cleaned successfully.\n")

    save_dataset(

        scheme_performance,

        "scheme_performance_clean.csv"

    )

    print("\nFirst four datasets completed successfully.\n")



    # ============================================================
    # CLEAN AUM HISTORY
    # ============================================================

    print("=" * 70)
    print("Cleaning AUM History")
    print("=" * 70)

    aum_history = remove_duplicates(aum_history)

    # Rename columns for consistency
    aum_history.rename(
        columns={
            "Scheme_Code": "Scheme Code",
            "Scheme_Name": "Scheme Name",
            "Fund_House": "Fund House"
        },
        inplace=True
    )

    # Convert Scheme Code
    aum_history["Scheme Code"] = pd.to_numeric(
        aum_history["Scheme Code"],
        errors="coerce"
    )

    # Convert Date
    aum_history["Date"] = pd.to_datetime(
        aum_history["Date"],
        errors="coerce"
    )

    # Convert AUM
    aum_history["AUM_Cr"] = pd.to_numeric(
        aum_history["AUM_Cr"],
        errors="coerce"
    )

    # Remove invalid rows
    aum_history.dropna(
        subset=[
            "Scheme Code",
            "Date",
            "AUM_Cr"
        ],
        inplace=True
    )

    aum_history["Scheme Code"] = (
        aum_history["Scheme Code"]
        .astype(int)
    )

    # Remove negative AUM
    aum_history = aum_history[
        aum_history["AUM_Cr"] > 0
    ]

    # Fill missing text columns
    aum_history["Scheme Name"] = (
        aum_history["Scheme Name"]
        .fillna("Unknown")
    )

    aum_history["Fund House"] = (
        aum_history["Fund House"]
        .fillna("Unknown")
    )

    # Sort values
    aum_history = aum_history.sort_values(
        [
            "Scheme Code",
            "Date"
        ]
    )

    print(aum_history.info())

    print("\nAUM History cleaned successfully.\n")

    save_dataset(
        aum_history,
        "aum_history_clean.csv"
    )

    # ============================================================
    # CLEAN SECTOR ALLOCATION
    # ============================================================

    print("=" * 70)
    print("Cleaning Sector Allocation")
    print("=" * 70)

    sector_allocation = remove_duplicates(
        sector_allocation
    )

    # Rename columns
    sector_allocation.rename(
        columns={
            "Scheme_Code": "Scheme Code",
            "Scheme_Name": "Scheme Name",
            "Fund_House": "Fund House"
        },
        inplace=True
    )

    # Convert Scheme Code
    sector_allocation["Scheme Code"] = pd.to_numeric(
        sector_allocation["Scheme Code"],
        errors="coerce"
    )

    # Convert Allocation %
    sector_allocation["Allocation_Percent"] = pd.to_numeric(
        sector_allocation["Allocation_Percent"],
        errors="coerce"
    )

    sector_allocation.dropna(
        subset=[
            "Scheme Code",
            "Allocation_Percent"
        ],
        inplace=True
    )

    sector_allocation["Scheme Code"] = (
        sector_allocation["Scheme Code"]
        .astype(int)
    )

    # Keep only valid percentages
    sector_allocation = sector_allocation[
        (sector_allocation["Allocation_Percent"] >= 0) &
        (sector_allocation["Allocation_Percent"] <= 100)
    ]

    # Fill missing text values
    sector_allocation["Scheme Name"] = (
        sector_allocation["Scheme Name"]
        .fillna("Unknown")
    )

    sector_allocation["Fund House"] = (
        sector_allocation["Fund House"]
        .fillna("Unknown")
    )

    sector_allocation["Sector"] = (
        sector_allocation["Sector"]
        .fillna("Unknown")
    )

    print(sector_allocation.info())

    print("\nSector Allocation cleaned successfully.\n")

    save_dataset(
        sector_allocation,
        "sector_allocation_clean.csv"
    )

    # ============================================================
    # VALIDATION SUMMARY
    # ============================================================

    print("=" * 70)
    print("VALIDATION SUMMARY")
    print("=" * 70)

    datasets = {
        "Fund Master": fund_master,
        "NAV History": nav_history,
        "Investor Transactions": investor_transactions,
        "Scheme Performance": scheme_performance,
        "AUM History": aum_history,
        "Sector Allocation": sector_allocation
    }

    for name, df in datasets.items():

        print(f"\n{name}")

        print("-" * 40)

        print(f"Rows    : {df.shape[0]}")

        print(f"Columns : {df.shape[1]}")

        print(f"Missing Values : {df.isnull().sum().sum()}")

    print("\n")

    print("=" * 70)
    print("All cleaned datasets saved successfully.")
    print("=" * 70)



    # ============================================================
    # ETL COMPLETED SUCCESSFULLY
    # ============================================================

    print("\n")
    print("=" * 70)
    print(" Bluestock Mutual Fund ETL Pipeline Completed Successfully ")
    print("=" * 70)

    print("\nProcessed files are available in:\n")
    print(PROCESSED_DIR)

    print("\nGenerated Files:")

    processed_files = [
        "fund_master_clean.csv",
        "nav_history_clean.csv",
        "investor_transactions_clean.csv",
        "scheme_performance_clean.csv",
        "aum_history_clean.csv",
        "sector_allocation_clean.csv"
    ]

    for file in processed_files:
        print(f"✓ {file}")

    print("\nThank you for using the Bluestock ETL Pipeline.")

# ============================================================
# ERROR HANDLING
# ============================================================

except FileNotFoundError as e:

    print("\n" + "=" * 70)
    print(" FILE NOT FOUND ERROR ")
    print("=" * 70)
    print(e)
    print("\nPlease check whether all required CSV files are present in:")
    print(RAW_DIR)

except PermissionError as e:

    print("\n" + "=" * 70)
    print(" PERMISSION ERROR ")
    print("=" * 70)
    print(e)
    print("\nPlease close any open CSV files and try again.")

except pd.errors.EmptyDataError as e:

    print("\n" + "=" * 70)
    print(" EMPTY CSV ERROR ")
    print("=" * 70)
    print(e)
    print("\nOne of the CSV files is empty.")

except Exception as e:

    print("\n" + "=" * 70)
    print(" ETL PIPELINE FAILED ")
    print("=" * 70)

    print(type(e).__name__)
    print(e)

    print("\nPlease check the error above and rerun the pipeline.")

# ============================================================
# END OF SCRIPT
# ============================================================

print("\n")
print("=" * 70)
print(" Script Finished ")
print("=" * 70)

 Bluestock Mutual Fund ETL Pipeline Started 


Reading Raw CSV Files
✓ All CSV files loaded successfully.

Fund Master Shape : (14208, 7)
NAV History Shape : (199336, 6)
Investor Transactions Shape : (25000, 10)
Scheme Performance Shape : (14208, 11)
AUM History Shape : (767232, 5)
Sector Allocation Shape : (127772, 5)


Cleaning Fund Master
Removed 0 duplicate rows
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14208 entries, 0 to 14207
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Scheme Code      14208 non-null  int64         
 1   Fund House       14208 non-null  object        
 2   Category         14208 non-null  object        
 3   Sub Category     14208 non-null  object        
 4   Scheme Name      14208 non-null  object        
 5   Net Asset Value  14208 non-null  float64       
 6   Date             14208 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(1